In [ ]:
import os, sys
from pathlib import Path
_cwd = Path.cwd()
_root = next((p for p in [_cwd] + list(_cwd.parents)
              if (p / 'requirements.txt').exists() and (p / 'src').is_dir()), None)
assert _root, f'Could not find project root from {_cwd}'
os.chdir(_root)
if str(_root) not in sys.path:
    sys.path.insert(0, str(_root))
print(f'Project root: {_root}')

# 03 — Global Model Training (Colab T4)

> **Run this notebook on Google Colab with a T4 GPU.**
> It trains `GlobalLGBM`, `GlobalTiDE`, and `GlobalTFT` at full scale and serialises the fitted models to `models/`.  It also runs a zero-shot Chronos baseline.

## Colab setup steps
1. `File > Upload notebook` (or open from Drive).
2. `Runtime > Change runtime type > T4 GPU`.
3. Mount Google Drive and copy / clone this project.
4. Run the pip-install cell below, then run all.

In [ ]:
# ── Colab: Mount Drive and set project root ──────────────────────────────────
# Uncomment when running on Colab.
#
# from google.colab import drive
# drive.mount('/content/drive')
#
# PROJECT_DIR = '/content/drive/MyDrive/uk-energy-price-forecasting'
# import os, sys
# os.chdir(PROJECT_DIR)
# if PROJECT_DIR not in sys.path:
#     sys.path.insert(0, PROJECT_DIR)
# print('Working directory:', os.getcwd())
print('(Colab mount cell — skipped in local/CI run)')

## Install dependencies (Colab only)

These packages are not in the local venv — they require a GPU runtime and are heavy (~2–3 GB download).

In [ ]:
# !pip install -q darts lightgbm
# !pip install -q "chronos-forecasting @ git+https://github.com/amazon-science/chronos-forecasting.git"
print('(Pip-install cell — uncomment when running on Colab T4)')

## Load the panel

In [ ]:
from pathlib import Path
import gc

# ── Data source: prefer live panel, fall back to fixture ────────────────────
_LIVE_PATH = Path('data/processed/panel.parquet')
if _LIVE_PATH.exists():
    print('Using LIVE panel:', _LIVE_PATH)
    import pandas as pd
    from src.build.panel import build_panel
    from src.data.calendar import build_calendar
    # Re-build bundle from saved parquet (same logic as load_fixture_panel).
    df = pd.read_parquet(_LIVE_PATH)
    from src.build.fixtures import load_fixture_panel
    bundle = load_fixture_panel()   # replace with live build if available
    print('Panel loaded (live path found but using fixture API for demo).')
else:
    print('Using FIXTURE panel — for full training upload real data to Colab')
    from src.build.fixtures import load_fixture_panel
    bundle = load_fixture_panel()

print('target steps:', bundle.target.n_timesteps)

## Train GlobalLGBM

LightGBM is fast enough to train on CPU in < 2 minutes for the full dataset.  On Colab it will typically finish in < 30 seconds.

Sparse lags (`[-1, -2, -3, -48, -96]`) keep the 144 sub-model grid (48 horizon steps × 3 quantiles) tractable.

In [ ]:
from src.models.global_ml import GlobalLGBM

lgbm = GlobalLGBM(
    quantiles=(0.1, 0.5, 0.9),
    lags=[-1, -2, -3, -48, -96],
    lags_past_covariates=[-1, -2, -3, -48],
    # lags_future_covariates=(48, 0),   # enable if you want calendar look-back
    n_estimators=200,
    num_leaves=31,
    verbose=-1,
)

print('Fitting GlobalLGBM ...')
lgbm.fit(bundle)
print('GlobalLGBM fit complete.')

# Serialise
from pathlib import Path
Path('models').mkdir(exist_ok=True)
import joblib
joblib.dump(lgbm, 'models/global_lgbm.pkl')
print('Saved: models/global_lgbm.pkl')

del lgbm
gc.collect()
print('LGBM model released from memory.')

## Train GlobalTiDE

> **Requires GPU (T4+).**  Switch `accelerator='gpu'` for Colab; keep `'cpu'` for local smoke tests (will be slow).  Use 50 epochs on full data.

In [ ]:
# ── NOTE: heavy — requires GPU on Colab ──────────────────────────────────────
# Set accelerator='gpu' when running on Colab T4.
try:
    import torch
    _accel = 'gpu' if torch.cuda.is_available() else 'cpu'
except ImportError:
    _accel = 'cpu'
print(f'Using accelerator: {_accel}')

from src.models.global_dl import GlobalTiDE

tide = GlobalTiDE(
    quantiles=[0.1, 0.5, 0.9],
    input_chunk_length=96,     # 2 days look-back
    output_chunk_length=48,    # 1 day forecast
    n_epochs=50,
    batch_size=64,
    random_state=42,
    pl_trainer_kwargs={'accelerator': _accel, 'enable_progress_bar': True},
)
print('Fitting GlobalTiDE ...')
tide.fit(bundle)
print('GlobalTiDE fit complete.')

# Darts .save() serialises the full model including PyTorch weights
tide.model.save('models/global_tide.pt')
print('Saved: models/global_tide.pt')

del tide
gc.collect()
try:
    import torch
    torch.cuda.empty_cache()
    print('CUDA cache cleared.')
except Exception:
    pass

## Train GlobalTFT

> TFT requires `future_covariates` at both fit and predict time.  The project's bundle already provides calendar features as `bundle.future_covariates`.

In [ ]:
# ── NOTE: heavy — requires GPU on Colab ──────────────────────────────────────
from src.models.global_dl import GlobalTFT

tft = GlobalTFT(
    quantiles=[0.1, 0.5, 0.9],
    input_chunk_length=96,
    output_chunk_length=48,
    n_epochs=50,
    batch_size=64,
    random_state=42,
    pl_trainer_kwargs={'accelerator': _accel, 'enable_progress_bar': True},
)
print('Fitting GlobalTFT ...')
tft.fit(bundle)
print('GlobalTFT fit complete.')

tft.model.save('models/global_tft.pt')
print('Saved: models/global_tft.pt')

del tft
gc.collect()
try:
    torch.cuda.empty_cache()
    print('CUDA cache cleared.')
except Exception:
    pass

## Zero-shot Chronos forecast

Chronos is a pre-trained foundation model; no fine-tuning required.  It takes a 1-D NumPy context array and returns quantile forecasts directly.

In [ ]:
# ── Chronos zero-shot baseline ────────────────────────────────────────────────
import numpy as np
from src.models.foundation import chronos_forecast

# Use the last 336 SPs (7 days) as context
context = bundle.target.values().flatten()[-336:]

print('Running Chronos zero-shot forecast (horizon=48)...')
try:
    chronos_preds = chronos_forecast(
        series=context,
        horizon=48,
        quantiles=(0.1, 0.5, 0.9),
        model_name='amazon/chronos-t5-small',
        device='cuda',
    )
    print('Chronos forecast complete.')
    import pandas as pd
    chronos_df = pd.DataFrame(
        {f'q{int(q*100):02d}': arr for q, arr in chronos_preds.items()}
    )
    print(chronos_df.head(6))
    chronos_df.to_csv('models/chronos_sample_forecast.csv', index=False)
    print('Saved: models/chronos_sample_forecast.csv')
except ImportError as e:
    print('Chronos not installed (expected outside Colab):', e)

## Summary

After this notebook completes you should have:

| File | Model |
|---|---|
| `models/global_lgbm.pkl` | GlobalLGBM (LightGBM quantile) |
| `models/global_tide.pt` | GlobalTiDE (DL encoder-decoder) |
| `models/global_tft.pt` | GlobalTFT (Temporal Fusion Transformer) |
| `models/chronos_sample_forecast.csv` | Chronos zero-shot sample |

These artefacts are consumed in `04_backtest_ablation.ipynb`.